In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

In [2]:
# get file paths
iter1_path = '../model_results/nn_fusion_clf_intensity_results/20260721_152222/intermediate_fusion_nn_results.csv'
iter2_path = '../model_results/nn_fusion_clf_intensity_results/20260721_153956/intermediate_fusion_nn_results.csv'
iter3_path = '../model_results/nn_fusion_clf_intensity_results/20260721_154247/intermediate_fusion_nn_results.csv'
iter4_path = '../model_results/nn_embedding_only_clf_intensity_results/20260721_154219/embedding_only_mlp_results.csv'
iter1_label = "Iteration 1"
iter2_label = "Added WeightRandomSampler"
iter3_label = "25 Features (MI)"
iter4_label = "Embeddings Only"
out_dir = './visualizations/intensity'

In [3]:
metrics = ['accuracy', 'f1', 'precision', 'recall', 'auc', 'specificity']
splits = ['train', 'val', 'test']
 
iter1 = pd.read_csv(iter1_path).set_index('embedding')
iter2 = pd.read_csv(iter2_path).set_index('embedding')
iter3 = pd.read_csv(iter3_path).set_index('embedding')
iter4 = pd.read_csv(iter4_path).set_index('embedding')

In [4]:
split = 'test'

all_iters = [iter1, iter2, iter3, iter4]
all_labels = [iter1_label, iter2_label, iter3_label, iter4_label]
all_colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

# only compare embedders present in every iteration
embedders_all4 = [e for e in iter1.index if all(e in it.index for it in all_iters)]
for label, it in zip(all_labels, all_iters):
    missing = [e for e in it.index if e not in embedders_all4]
    if missing:
        print(f"Note: in {label} only (excluded from comparison): {missing}")

x4 = np.arange(len(embedders_all4))
width4 = 0.8 / len(all_iters)  # split available width across 4 bars

for metric in metrics:
    col = f'{split}_{metric}'

    # skip metrics not present in every iteration's csv (e.g. specificity added later)
    missing_iters = [label for label, it in zip(all_labels, all_iters) if col not in it.columns]
    if missing_iters:
        print(f"Skipping {metric}: missing in {missing_iters}")
        continue

    fig, ax = plt.subplots(figsize=(12, 5))

    values = [it.loc[embedders_all4, col].values for it in all_iters]

    for i, (v, label, color) in enumerate(zip(values, all_labels, all_colors)):
        offset = (i - (len(all_iters) - 1) / 2) * width4
        ax.bar(x4 + offset, v, width4, label=label, color=color)

    # default 0.75-0.95, shift both bounds down by 0.05 steps until min value fits
    data_min = min(v.min() for v in values)
    ylim_low, ylim_high = 0.75, 1.0
    while data_min < ylim_low:
        ylim_low -= 0.05
    ax.set_ylim(ylim_low, ylim_high)

    ax.set_ylabel(metric.upper())
    ax.set_title(f'{split.capitalize()} {metric.upper()}: All Iterations', fontsize=13)
    ax.grid(axis='y', alpha=0.3)
    ax.set_xticks(x4)
    ax.set_xticklabels(embedders_all4, rotation=30, ha='right')
    ax.legend(loc='lower right')

    fig.tight_layout()
    fig.savefig(f'{out_dir}/all_iterations_{split}_{metric}.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved {split} {metric} comparison plot (all iterations).")

Saved test accuracy comparison plot (all iterations).
Saved test f1 comparison plot (all iterations).
Saved test precision comparison plot (all iterations).
Saved test recall comparison plot (all iterations).
Saved test auc comparison plot (all iterations).
Saved test specificity comparison plot (all iterations).


In [5]:
# only compare embedders present in both iterations
embedders = [e for e in iter1.index if e in iter2.index]
missing_1 = [e for e in iter2.index if e not in iter1.index]
missing_2 = [e for e in iter1.index if e not in iter2.index]
if missing_1:
    print(f"Note: in {iter2_label} only: {missing_1}")
if missing_2:
    print(f"Note: in {iter1_label} only: {missing_2}")
 
os.makedirs(out_dir, exist_ok=True)
 
x = np.arange(len(embedders))
width = 0.35

In [6]:
# per-metric bar comparisons for test split
split = 'test'
for metric in metrics:
    col = f'{split}_{metric}'
    fig, ax = plt.subplots(figsize=(10, 5))

    v1 = iter2.loc[embedders, col].values
    v2 = iter3.loc[embedders, col].values

    ax.bar(x - width/2, v1, width, label=iter2_label, color='#4C72B0')
    ax.bar(x + width/2, v2, width, label=iter3_label, color='#DD8452')

    # default 0.75-0.95, shift both bounds down by 0.05 steps until min value fits
    data_min = min(v1.min(), v2.min())
    ylim_low, ylim_high = 0.75, 0.95
    while data_min < ylim_low:
        ylim_low -= 0.05
        ylim_high -= 0.05
    ax.set_ylim(ylim_low, ylim_high)

    ax.set_ylabel(metric.upper())
    ax.set_title(f'{split.capitalize()} {metric.upper()}: {iter2_label} vs {iter3_label}', fontsize=13)
    ax.grid(axis='y', alpha=0.3)
    ax.set_xticks(x)
    ax.set_xticklabels(embedders, rotation=30, ha='right')
    ax.legend(loc='lower right')

    fig.tight_layout()
    fig.savefig(f'{out_dir}/all_extra_vs_just_calvados_ah_pairs_{split}_{metric}.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved {split} {metric} comparison plot.")

Saved test accuracy comparison plot.
Saved test f1 comparison plot.
Saved test precision comparison plot.
Saved test recall comparison plot.
Saved test auc comparison plot.
Saved test specificity comparison plot.
